Comprueba que el resumen exista

In [22]:
from pathlib import Path
print(Path("../data/fasta_resumen.csv").exists())

import pandas as pd
from Bio import SeqIO

filas = []
for rec in SeqIO.parse("../data/PanTEon_Database_v1.6.1.fasta", "fasta"):
    id_, resto = rec.description.split("#", 1)
    clasif, especie = resto.split("@", 1)
    seq = str(rec.seq).upper()
    filas.append({"id": id_.strip(), "clasificacion": clasif.strip(),
                  "niveles": clasif.count("/") + 1, "especie": especie.strip(),
                  "longitud": len(seq), "n_N": seq.count("N")})
pd.DataFrame(filas).to_csv("../data/fasta_resumen.csv", index=False)
print(len(filas), "registros")

True
239946 registros


carga el CSV limpio y ve la clave de cruce

In [23]:
import sys; sys.path.append("..")
import pandas as pd
from pipeline.limpiar import limpiar_csv, clave

fa = pd.read_csv("../data/fasta_resumen.csv", keep_default_na=False)
d = limpiar_csv("../data/PanTEon_Database_metadata_v1.6.1.csv")

fa["species_key"] = fa["especie"].map(clave)
fa["pct_N"] = fa["n_N"] / fa["longitud"] * 100
fa["no_pdb"] = ~fa["id"].str.startswith("PDB")
print(fa.shape, d.shape)

(239946, 9) (2790, 11)


fila por especie

In [24]:
res = fa.groupby("species_key").agg(
    n_seq=("id", "size"),
    con_N=("n_N", lambda s: (s > 0).sum()),
    pct_N_max=("pct_N", "max"),
    ids_no_pdb=("no_pdb", "sum"),
)
meta = d.set_index("species_key")[["Kingdom", "Phylum", "Class", "Order", "Family", "Source"]]
res = res.join(meta)
res["tax_nulos"] = res[["Phylum", "Class", "Order", "Family"]].isna().sum(axis=1)
res = res[res["n_seq"].between(20, 300)]
print(len(res), "especies candidatas")

1022 especies candidatas


separa a las candidatas en los tres tipos: limpia, con N e ID no PDB, y con taxonomía incompleta.

In [25]:
cols = ["n_seq", "con_N", "pct_N_max", "ids_no_pdb", "tax_nulos", "Kingdom"]

print("--- Limpias ---")
print(res[(res.con_N == 0) & (res.ids_no_pdb == 0) & (res.tax_nulos == 0)][cols]
      .groupby("Kingdom").head(3))

print("--- Con N e IDs no PDB ---")
print(res[(res.con_N > 0) & (res.ids_no_pdb > 0)][cols].head(8))

print("--- Taxonomía incompleta ---")
print(res[res.tax_nulos > 0][cols].head(8))

--- Limpias ---
                        n_seq  con_N  pct_N_max  ids_no_pdb  tax_nulos   Kingdom
species_key                                                                     
amylostereum areolatum     27      0        0.0           0          0     Fungi
armillaria cepistipes      28      0        0.0           0          0     Fungi
ascochyta koolunga         57      0        0.0           0          0     Fungi
globodera pallida          25      0        0.0           0          0  Animalia
lepidurus packardi         90      0        0.0           0          0  Animalia
utricularia gibba          25      0        0.0           0          0   Plantae
xylocarpus granatum        37      0        0.0           0          0   Plantae
--- Con N e IDs no PDB ---
                        n_seq  con_N  pct_N_max  ids_no_pdb  tax_nulos   Kingdom
species_key                                                                     
acomys cahirinus           39     26   1.266805          39       

Exploración extra: especies candidatas con taxonomía incompleta

In [26]:
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

cols = ["n_seq", "con_N", "ids_no_pdb", "tax_nulos", "Kingdom"]

inc = res[res.tax_nulos > 0].copy()
inc["falta"] = inc[["Phylum", "Class", "Order", "Family"]].isna().apply(
    lambda r: ",".join(r.index[r]), axis=1)
print(len(inc), "especies candidatas con taxonomía incompleta")
print(inc[cols + ["falta"]].sort_values("n_seq", ascending=False).head(15).to_string())

print("\n--- Con N e IDs no PDB ---")
print(res[(res.con_N > 0) & (res.ids_no_pdb > 0)][cols]
      .sort_values("n_seq", ascending=False).head(8).to_string())

60 especies candidatas con taxonomía incompleta
                                     n_seq  con_N  ids_no_pdb  tax_nulos   Kingdom   falta
species_key                                                                               
phytophthora infestans                 277    168           0          1     Fungi   Class
eragrostis curvula                     276     33           0          1   Plantae   Class
anisus vortex                          274     11           0          1  Animalia   Order
brachypodium distachyon                223     23           0          1   Plantae   Class
ampullaceana balthica                  211      9           0          1  Animalia   Order
argyrosomus regius                     201     25           0          1  Animalia   Order
citrus clementina                      191     13           0          1   Plantae   Class
aquilegia eximia                       186     23           0          1   Plantae   Class
abrus pulchellus subsp cantoniensis    178

Exploración extra: nombres de una sola palabra en el CSV

In [27]:
una = d[~d["species_key"].str.contains(" ")]
print(len(una), "nombres de una sola palabra en el CSV")
print(una[["Species", "Kingdom", "Phylum", "Class", "Order", "Family"]].head(15).to_string())

45 nombres de una sola palabra en el CSV
               Species   Kingdom    Phylum        Class          Order      Family
39          Afrotheria  Animalia  Chordata     Mammalia           None        None
110            Amniota  Animalia  Chordata         None           None        None
243       Artiodactyla  Animalia  Chordata     Mammalia   Artiodactyla        None
521      Boreoeutheria  Animalia  Chordata     Mammalia           None        None
630          Carnivora  Animalia  Chordata     Mammalia      Carnivora        None
641         Catarrhini  Animalia  Chordata     Mammalia       Primates        None
705         Chiroptera  Animalia  Chordata     Mammalia     Chiroptera        None
725          Chrysemys  Animalia  Chordata         None     Testudines    Emydidae
929              Danio  Animalia  Chordata  Actinopteri  Cypriniformes  Danionidae
931         Danionidae  Animalia  Chordata  Actinopteri  Cypriniformes  Danionidae
1021    Durocryptodira  Animalia  Chordata    

Exploración extra: textos placeholder ("No ...") y relación entre IDs no PDB, Source y Kingdom

In [28]:
tax = ["Kingdom", "Phylum", "Class", "Order", "Family"]
for c in tax:
    ph = d[c][d[c].fillna("").str.match(r"(?i)^no\b")]
    print(c, "->", ph.value_counts().to_dict())

elegidas = ["armillaria cepistipes", "acomys cahirinus", "brachypodium distachyon"]
print(d[d.species_key.isin(elegidas)][["Species", "Kingdom", "Phylum", "Class", "Order", "Family", "Source"]].to_string())

tmp = fa.groupby("species_key")["no_pdb"].mean().gt(0).rename("algun_no_pdb").to_frame()
tmp = tmp.join(d.set_index("species_key")[["Source", "Kingdom"]])
print(pd.crosstab(tmp["algun_no_pdb"], tmp["Source"]))
print(pd.crosstab(tmp["algun_no_pdb"], tmp["Kingdom"]))

Kingdom -> {}
Phylum -> {}
Class -> {}
Order -> {}
Family -> {}
                     Species   Kingdom         Phylum           Class       Order           Family   Source
13          Acomys cahirinus  Animalia       Chordata        Mammalia    Rodentia          Muridae     Dfam
231    Armillaria cepistipes     Fungi  Basidiomycota  Agaricomycetes  Agaricales  Physalacriaceae  RM2+MCH
547  Brachypodium distachyon   Plantae    Angiosperma            None      Poales          Poaceae  RM2+MCH
Source        Dfam  RM2+MCH
algun_no_pdb               
False            0     2403
True           386        1
Kingdom       Animalia  Fungi  Plantae
algun_no_pdb                          
False              395   1558      450
True               385      1        1


Prueba A: `separar_encabezado` con un encabezado válido y uno mal formado

In [29]:
from pipeline.fasta import separar_encabezado, leer_fasta

e = separar_encabezado("PDB00000001#CLASSII/HELITRON/HELITRON @Halicephalobus mephisto")
assert e["sequence_id"] == "PDB00000001"
assert (e["te_class"], e["te_order"], e["te_superfamily"]) == ("CLASSII", "HELITRON", "HELITRON")
assert e["species_key"] == "halicephalobus mephisto"
print(e)

try:
    separar_encabezado("sin_formato")
except ValueError as err:
    print("OK, rechazó:", err)

{'sequence_id': 'PDB00000001', 'classification_raw': 'CLASSII/HELITRON/HELITRON', 'te_class': 'CLASSII', 'te_order': 'HELITRON', 'te_superfamily': 'HELITRON', 'species_raw': 'Halicephalobus mephisto', 'species_key': 'halicephalobus mephisto'}
OK, rechazó: Encabezado mal formado: 'sin_formato'


Prueba B: `leer_fasta` con las 3 especies del prototipo

In [30]:
ESPECIES = {"armillaria cepistipes", "acomys cahirinus", "brachypodium distachyon"}
docs = list(leer_fasta("../data/PanTEon_Database_v1.6.1.fasta", ESPECIES))
df = pd.DataFrame(docs)
print(len(docs))
print(df.groupby("species_key").size())
print(df.groupby(["te_class", "te_order", "te_superfamily"]).size())

290
species_key
acomys cahirinus            39
armillaria cepistipes       28
brachypodium distachyon    223
dtype: int64
te_class  te_order    te_superfamily
CLASSI    LINE        L1                87
                      RTE                2
                      RTEX               1
                      TAD1               1
          LTR         COPIA             39
                      ERV               28
                      GYPSY             62
                      LARD               3
                      TRIM               3
          RETROPOSON  RETROPOSON         2
          SINE        ALU                1
                      B2                 5
                      SINE               4
CLASSII   HELITRON    HELITRON           3
          MITE        MITE              30
          TIR         CMC                6
                      HAT                2
                      MULE               2
                      PIFHARBINGER       4
                      TC

Prueba C: el lector contra el resumen de la exploración

In [31]:
chk = df.merge(fa[["id", "longitud", "n_N"]], left_on="sequence_id", right_on="id")
print(len(chk),
      (chk["sequence"].str.len() == chk["longitud"]).all(),
      (chk["sequence"].str.count("N") == chk["n_N"]).all())

290 True True


Prueba D: `metricas` con casos a mano

In [32]:
from pipeline.kmers import metricas

m = metricas("ATGCA")                       # G y C = 2 de 5
assert m["length"] == 5 and m["n_count"] == 0
assert abs(m["gc_content"] - 40.0) < 1e-9

m = metricas("ATGNCA")                      # G y C = 2 de 5 bases conocidas
assert m["length"] == 6 and m["n_count"] == 1
assert abs(m["gc_content"] - 40.0) < 1e-9   # sobre la longitud daría 33.3
assert abs(m["n_pct"] - 100 / 6) < 1e-9

try:
    metricas("NNNN")
except ValueError as err:
    print("OK, rechazó:", err)

OK, rechazó: Secuencia sin bases conocidas: no se puede calcular el GC


Prueba E: métricas de las 290 secuencias contra Biopython

In [33]:
from Bio.SeqUtils import gc_fraction

filas = []
for doc in docs:
    m = metricas(doc["sequence"])
    m["id"] = doc["sequence_id"]
    m["gc_biopython"] = gc_fraction(doc["sequence"]) * 100
    filas.append(m)
met = pd.DataFrame(filas)

chk2 = met.merge(fa[["id", "longitud", "n_N"]], on="id")
print(len(chk2),
      (chk2["length"] == chk2["longitud"]).all(),
      (chk2["n_count"] == chk2["n_N"]).all(),
      ((chk2["gc_content"] - chk2["gc_biopython"]).abs() < 1e-9).all())

290 True True True


Prueba F: diferencia entre GC sobre bases conocidas y GC sobre la longitud total

In [34]:
met["gc_sobre_longitud"] = met["gc_content"] * (met["length"] - met["n_count"]) / met["length"]
met["dif"] = met["gc_content"] - met["gc_sobre_longitud"]
print(met["dif"].describe().round(3))
print(met.loc[met["dif"].idxmax(), ["id", "n_pct", "gc_content", "gc_sobre_longitud"]])
print(met[["length", "gc_content", "n_pct"]].describe().round(2))

count    290.000
mean       0.157
std        0.613
min       -0.000
25%        0.000
50%        0.000
75%        0.000
max        4.519
Name: dif, dtype: float64
id                   PDB00053691
n_pct                   7.682119
gc_content             58.823529
gc_sobre_longitud      54.304636
Name: 201, dtype: object
         length  gc_content   n_pct
count    290.00      290.00  290.00
mean    3485.23       48.21    0.31
std     2969.79        6.90    1.17
min      111.00       28.57    0.00
25%      743.50       44.02    0.00
50%     3315.50       47.52    0.00
75%     5201.00       52.34    0.00
max    13817.00       70.27    7.83


Prueba G: `contar_kmers` con casos a mano

In [35]:
import importlib, pipeline.kmers as pk
importlib.reload(pk)
from pipeline.kmers import contar_kmers, kmers_todos, etiquetas

def como_dict(arr, k):
    return {e: int(c) for e, c in zip(etiquetas(k), arr) if c}

# Ejemplo del enunciado: ATGCA
assert como_dict(contar_kmers("ATGCA", 1), 1) == {"A": 2, "T": 1, "G": 1, "C": 1}
assert como_dict(contar_kmers("ATGCA", 2), 2) == {"AT": 1, "TG": 1, "GC": 1, "CA": 1}
assert como_dict(contar_kmers("ATGCA", 3), 3) == {"ATG": 1, "TGC": 1, "GCA": 1}

# Con N: se saltan GN y NC, y TGN, GNC, NCA
assert como_dict(contar_kmers("ATGNCA", 1), 1) == {"A": 2, "T": 1, "G": 1, "C": 1}
assert como_dict(contar_kmers("ATGNCA", 2), 2) == {"AT": 1, "TG": 1, "CA": 1}
assert como_dict(contar_kmers("ATGNCA", 3), 3) == {"ATG": 1}

# Bordes
assert como_dict(contar_kmers("AAAA", 2), 2) == {"AA": 3}     # se solapan
assert contar_kmers("AT", 3).sum() == 0                       # k mayor que la secuencia
assert len(contar_kmers("AT", 3)) == 64
assert sum(len(v) for v in kmers_todos("ACGTACGT").values()) == 5460

try:
    contar_kmers("atg", 1)
except ValueError as err:
    print("OK, rechazó minúsculas:", err)

OK, rechazó minúsculas: Caracteres no válidos: se esperan solo A, C, G, T y N en mayúsculas


Prueba H: k-mers de las 290 secuencias contra un conteo independiente

In [36]:
from collections import Counter

errores = 0
for doc in docs:
    s = doc["sequence"]; L = len(s)
    for k in range(1, 7):
        ref = Counter(s[i:i+k] for i in range(L - k + 1) if "N" not in s[i:i+k])
        if como_dict(contar_kmers(s, k), k) != dict(ref):
            errores += 1
print("Secuencia x k con diferencias:", errores, "de", len(docs) * 6)

Secuencia x k con diferencias: 0 de 1740


Prueba I: la suma de k-mers cuadra con la longitud

In [37]:
malas = 0
for doc in docs:
    s = doc["sequence"]; L = len(s)
    total = {k: int(contar_kmers(s, k).sum()) for k in range(1, 7)}
    if "N" not in s:
        malas += any(total[k] != L - k + 1 for k in total)   # sin N: exactamente L-k+1
    malas += total[1] != L - s.count("N")                    # k=1: las bases conocidas
    malas += any(total[k] > L - k + 1 for k in total)        # nunca más de L-k+1
print("Incumplimientos:", malas)

Incumplimientos: 0


Prueba J: velocidad del conteo y estimación para toda la base

In [38]:
import time
t0 = time.perf_counter()
for doc in docs:
    kmers_todos(doc["sequence"])
t = time.perf_counter() - t0

nt_proto = sum(len(d["sequence"]) for d in docs)
nt_total = int(fa["longitud"].sum())
print(f"{t:.2f} s para {nt_proto:,} nt")
print(f"Estimado para toda la base ({nt_total:,} nt): {t * nt_total / nt_proto / 60:.1f} min")  

0.10 s para 1,010,716 nt
Estimado para toda la base (963,901,415 nt): 1.6 min


Carga K: conexión a MongoDB y creación de `panteon_proto` con el esquema real

In [45]:
import importlib, pipeline.carga as pc
importlib.reload(pc)
from pymongo import MongoClient
from database.setup_db import setup_database

client = MongoClient("mongodb://localhost:27017")
setup_database(db_name="panteon_proto", drop_existing=True)

=== Inicializando Base de Datos: 'panteon_proto' en mongodb://localhost:27017 ===
[*] Creando colección 'species' con validación $jsonSchema...
[*] Creando colección 'sequences' con validación $jsonSchema...

=== Configurando Índices Estratégicos ===
[*] Índices en colección 'species'...
[*] Índices en colección 'sequences'...

✓ Configuración completada con éxito.

--- Índices actuales en 'species' ---
 - _id_: SON([('_id', 1)])
 - idx_species_key_unique: SON([('species_key', 1)])
 - idx_kingdom_class: SON([('Kingdom', 1), ('Class', 1)])
 - idx_species_source: SON([('Source', 1)])

--- Índices actuales en 'sequences' ---
 - _id_: SON([('_id', 1)])
 - idx_sequence_id_unique: SON([('sequence_id', 1)])
 - idx_species_superfamily: SON([('species_key', 1), ('te_superfamily', 1)])
 - idx_te_hierarchy: SON([('te_class', 1), ('te_order', 1), ('te_superfamily', 1)])
 - idx_metrics_length: SON([('metrics.length', 1)])
 - idx_metrics_gc: SON([('metrics.gc_content', 1)])


Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'panteon_proto')

Carga L: inserción en `panteon_proto` (con el esquema real)

In [46]:
from collections import Counter
db = client["panteon_proto"]

n_sp, rech_sp = pc.cargar_lotes(db.species, pc.armar_especies(d, ESPECIES), "species_key")
n_sq, rech_sq = pc.cargar_lotes(db.sequences, (pc.armar_secuencia(c) for c in docs), "sequence_id")

print("species:   insertadas", n_sp, "| rechazadas", rech_sp)
print("sequences: insertadas", n_sq, "| rechazadas", len(rech_sq),
      Counter(code for _, code in rech_sq))
print("Primeros IDs rechazados:", [i for i, _ in rech_sq[:3]])

species:   insertadas 3 | rechazadas []
sequences: insertadas 290 | rechazadas 0 Counter()
Primeros IDs rechazados: []


Carga M: inserción en `panteon_proto_libre` (sin validador)

In [47]:
client.drop_database("panteon_proto_libre")     # solo esta base de prototipo
libre = client["panteon_proto_libre"]

n_sp2, r1 = pc.cargar_lotes(libre.species, pc.armar_especies(d, ESPECIES), "species_key")
n_sq2, r2 = pc.cargar_lotes(libre.sequences, (pc.armar_secuencia(c) for c in docs), "sequence_id")
print("species:", n_sp2, "| sequences:", n_sq2, "| rechazos:", len(r1) + len(r2))

species: 3 | sequences: 290 | rechazos: 0


Carga N: comprobaciones de la carga (conteos, huérfanas y tamaño)

In [48]:
print("Leídas:", len(docs), "| en la base libre:", libre.sequences.count_documents({}))

huerfanas = set(db.sequences.distinct("species_key")) - set(db.species.distinct("species_key"))
print("En panteon_proto, species_key sin especie:", huerfanas)

st = libre.command("collStats", "sequences")
print(f"Tamaño medio por documento: {st['avgObjSize']/1024:.1f} KB")
print(f"Estimado para 239.946 secuencias sin comprimir: {st['avgObjSize']*239946/1024**3:.1f} GB")
print(f"En disco (comprimido por MongoDB), muestra: {st['storageSize']/1024**2:.1f} MB para {st['count']} docs")

Leídas: 290 | en la base libre: 290
En panteon_proto, species_key sin especie: set()
Tamaño medio por documento: 33.9 KB
Estimado para 239.946 secuencias sin comprimir: 7.8 GB
En disco (comprimido por MongoDB), muestra: 0.0 MB para 290 docs


Verificación O: relectura desde MongoDB contra un conteo independiente

In [49]:
from collections import Counter

libre = client["panteon_proto_libre"]
S = libre.sequences
orig = {x["sequence_id"]: x["sequence"] for x in docs}

# una por especie, la que tiene más N, y 10 al azar (semilla fija)
import random
por_especie = {}
for x in docs:
    por_especie.setdefault(x["species_key"], x["sequence_id"])
mas_N = max(docs, key=lambda x: x["sequence"].count("N") / len(x["sequence"]))["sequence_id"]
ids = list(por_especie.values()) + [mas_N] + random.Random(1).sample(list(orig), 10)

def ref_kmers(s, k):
    return dict(Counter(s[i:i+k] for i in range(len(s) - k + 1) if "N" not in s[i:i+k]))

difs = 0
for sid in ids:
    g = S.find_one({"sequence_id": sid})
    difs += g["sequence"] != orig[sid]
    for k in range(1, 7):
        difs += g["kmers"][f"k{k}"] != ref_kmers(orig[sid], k)
print("Secuencias revisadas:", len(ids), "| diferencias:", difs)

Secuencias revisadas: 14 | diferencias: 0


Verificación P: tipos de datos guardados y nulos en `species`

In [26]:
# Tipos vistos desde Python (int = entero, float = decimal)
malos = 0
for g in S.find({}, {"kmers": 1, "metrics": 1}):
    for dic in g["kmers"].values():
        malos += sum(type(v) is not int for v in dic.values())
    m = g["metrics"]
    malos += type(m["length"]) is not int
    malos += type(m["n_count"]) is not int
    malos += type(m["gc_content"]) is not float
print("Valores con tipo inesperado:", malos)

# Tipos vistos por el servidor
print("gc_content que no es double:", S.count_documents({"metrics.gc_content": {"$not": {"$type": "double"}}}))

# Nulos en species
sp = libre.species
print("Class null explícito:", sp.count_documents({"Class": {"$type": "null"}}))
print("Class ausente:", sp.count_documents({"Class": {"$exists": False}}))
print("Class null o ausente:", sp.count_documents({"Class": None}))
print(sp.find_one({"species_key": "brachypodium distachyon"},
                  {"_id": 0, "Species": 1, "Class": 1, "source_2": 1, "annotation": 1}))

Valores con tipo inesperado: 0
gc_content que no es double: 0
Class null explícito: 1
Class ausente: 0
Class null o ausente: 1
{'Class': None, 'Species': 'Brachypodium distachyon', 'source_2': 'Repbase', 'annotation': None}


Verificación Q: consulta de k-mers con proyección (k = 2 y k = 3)

In [27]:
import bson

sid = por_especie["armillaria cepistipes"]       # sin N
L = len(orig[sid])
for k in (2, 3):
    r = S.find_one({"sequence_id": sid}, {"_id": 0, "sequence_id": 1, f"kmers.k{k}": 1})
    perfil = r["kmers"][f"k{k}"]
    print(f"k={k}: distintos {len(perfil)} | suma {sum(perfil.values())} | L-k+1 = {L-k+1}")
    print("  más frecuentes:", sorted(perfil.items(), key=lambda t: -t[1])[:5])

completo = S.find_one({"sequence_id": sid})
parcial = S.find_one({"sequence_id": sid}, {"_id": 0, "kmers.k3": 1})
print(f"Documento completo: {len(bson.encode(completo))/1024:.1f} KB | solo k3: {len(bson.encode(parcial))/1024:.2f} KB")

k=2: distintos 16 | suma 4815 | L-k+1 = 4815
  más frecuentes: [('GA', 397), ('AA', 378), ('CA', 363), ('AC', 358), ('AT', 334)]
k=3: distintos 64 | suma 4814 | L-k+1 = 4814
  más frecuentes: [('GAA', 142), ('CAA', 116), ('GGA', 115), ('AAG', 112), ('AGA', 107)]
Documento completo: 48.3 KB | solo k3: 0.59 KB


Verificación R: efecto del índice y tamaño de la colección

In [30]:
sid = por_especie["armillaria cepistipes"]   # sin N, y está en las dos bases

for nombre in ("panteon_proto", "panteon_proto_libre"):
    r = client[nombre].command({
        "explain": {"find": "sequences", "filter": {"sequence_id": sid}},
        "verbosity": "executionStats",
    })
    ex = r["executionStats"]
    print(f"{nombre}: claves examinadas {ex['totalKeysExamined']} | documentos examinados {ex['totalDocsExamined']}")

st = libre.command("collStats", "sequences")
print(f"Sin comprimir: {st['size']/1024**2:.1f} MB | en disco: {st['storageSize']/1024**2:.1f} MB | índices: {st['totalIndexSize']/1024**2:.2f} MB")

panteon_proto: claves examinadas 1 | documentos examinados 1
panteon_proto_libre: claves examinadas 0 | documentos examinados 290
Sin comprimir: 9.6 MB | en disco: 3.8 MB | índices: 0.02 MB
